# CUDA Kernel 面试主线 · 第 12/12 课：ArgMax 对归约与确定性

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：归约 `(value,index)` 对，处理相等值、NaN 和两阶段输出。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：ArgMax 不只归约值，还要同步携带索引；这是自定义归约算子的典型模式。

## 核心心智模型

### 1. 它是什么，解决什么问题

ArgMax 不只归约值，还要同步携带索引；这是自定义归约算子的典型模式。

### 2. 它如何工作

每线程找局部 `(val,idx)`，warp/block combine 时按值选择，并用索引规则打破平局，最后二阶段合并。

### 3. 正确性条件与常见误区

必须定义 tie-break 和 NaN policy；否则并行归约顺序变化会令结果不确定。当前初版采用严格 `>`，相等时保留先到者。

### 4. 性能与工程取舍

保证最小索引确定性会增加比较逻辑；若调用方只关心任一最大值可放宽。

## 具体演示

输入 [5,7,7]：若约定最小索引，答案必须是 1；不同 tree 顺序不能改变它。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐确定性比较条件；注意初始化 idx=-1。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/12_argmax.cu
#include <cuda_runtime.h>
#include <float.h>
#include <stdio.h>
#include <stdlib.h>

namespace {

// ArgMax: 找到 input[0:n] 中最大值的索引。
//
// 面试重点：
// 1. reduce 时同时传递 (value, index) 对。
// 2. warp shuffle 传 value 比较，再传 index 选择。
// 3. 两轮 kernel：第一轮每个 block 出一个 (val, idx)，第二轮归并。

struct ValIdx {
    float val;
    int idx;
};

template<int BLOCK_SIZE>
__device__ __forceinline__ ValIdx warp_reduce_max(ValIdx vi) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset >>= 1) {
        float other_val = __shfl_down_sync(0xffffffff, vi.val, offset);
        int other_idx = __shfl_down_sync(0xffffffff, vi.idx, offset);
        if (______) {  // TODO: 选择更大值；相等时选择更小有效索引
            vi.val = other_val;
            vi.idx = other_idx;
        }
    }
    return vi;
}

template<int BLOCK_SIZE>
__device__ __forceinline__ ValIdx block_reduce_max(ValIdx vi) {
    constexpr int NUM_WARPS = (BLOCK_SIZE + 31) / 32;
    __shared__ float smem_val[NUM_WARPS];
    __shared__ int smem_idx[NUM_WARPS];

    int lane = threadIdx.x & 31;
    int warp = threadIdx.x >> 5;

    vi = warp_reduce_max<BLOCK_SIZE>(vi);

    if (lane == 0) {
        smem_val[warp] = vi.val;
        smem_idx[warp] = vi.idx;
    }
    __syncthreads();

    if (warp == 0) {
        vi.val = (lane < NUM_WARPS) ? smem_val[lane] : -FLT_MAX;
        vi.idx = (lane < NUM_WARPS) ? smem_idx[lane] : -1;
        vi = warp_reduce_max<BLOCK_SIZE>(vi);
    }

    return vi;
}

// 第一轮：每个 block 用 grid-stride loop 找到局部最大值及其索引
template<int BLOCK_SIZE>
__global__ void argmax_kernel(
    const float* __restrict__ input,
    float* __restrict__ partial_val,
    int* __restrict__ partial_idx,
    int n
) {
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + tid;
    int stride = blockDim.x * gridDim.x;

    // 每个线程先通过 grid-stride loop 找到自己负责的元素中的最大值
    ValIdx local = {-FLT_MAX, -1};
    for (int i = idx; i < n; i += stride) {
        if (input[i] > local.val) {
            local.val = input[i];
            local.idx = i;
        }
    }

    // block 级 reduce
    ValIdx result = block_reduce_max<BLOCK_SIZE>(local);

    if (tid == 0) {
        partial_val[blockIdx.x] = result.val;
        partial_idx[blockIdx.x] = result.idx;
    }
}

// 第二轮：把所有 block 的 partial 结果归并，只需一个 block
template<int BLOCK_SIZE>
__global__ void argmax_final_kernel(
    const float* __restrict__ partial_val,
    const int* __restrict__ partial_idx,
    int* __restrict__ output,
    int num_blocks
) {
    int tid = threadIdx.x;

    ValIdx local = {-FLT_MAX, -1};
    // 一个 block 处理所有 partial，stride = BLOCK_SIZE
    for (int i = tid; i < num_blocks; i += BLOCK_SIZE) {
        if (partial_val[i] > local.val) {
            local.val = partial_val[i];
            local.idx = partial_idx[i];
        }
    }

    ValIdx result = block_reduce_max<BLOCK_SIZE>(local);

    if (tid == 0) {
        *output = result.idx;
    }
}

} // namespace

// ===================== Host 调用接口 =====================

void launch_argmax(
    const float* d_input,
    int* d_output,
    float* d_partial_val,
    int* d_partial_idx,
    int n,
    int num_blocks,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    argmax_kernel<BLOCK_SIZE><<<num_blocks, BLOCK_SIZE, 0, stream>>>(
        d_input, d_partial_val, d_partial_idx, n
    );
    argmax_final_kernel<BLOCK_SIZE><<<1, BLOCK_SIZE, 0, stream>>>(
        d_partial_val, d_partial_idx, d_output, num_blocks
    );
}

// ===================== 测试 =====================

int main() {
    const int N = 1 << 20;  // 1M 元素
    const int NUM_BLOCKS = 128;

    // Host 数据
    float* h_input = (float*)malloc(N * sizeof(float));
    srand(42);
    int expected_idx = 0;
    float expected_val = -FLT_MAX;
    for (int i = 0; i < N; i++) {
        h_input[i] = (float)rand() / RAND_MAX;
        if (h_input[i] > expected_val) {
            expected_val = h_input[i];
            expected_idx = i;
        }
    }

    // Device 数据
    float *d_input, *d_partial_val;
    int *d_partial_idx, *d_output;
    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_partial_val, NUM_BLOCKS * sizeof(float));
    cudaMalloc(&d_partial_idx, NUM_BLOCKS * sizeof(int));
    cudaMalloc(&d_output, sizeof(int));

    cudaMemcpy(d_input, h_input, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行
    launch_argmax(d_input, d_output, d_partial_val, d_partial_idx, N, NUM_BLOCKS, 0);

    // 取结果
    int h_output;
    cudaMemcpy(&h_output, d_output, sizeof(int), cudaMemcpyDeviceToHost);

    printf("ArgMax result: index = %d, value = %f\n", h_output, h_input[h_output]);
    printf("Expected:      index = %d, value = %f\n", expected_idx, expected_val);
    printf("%s\n", (h_output == expected_idx) ? "PASS" : "FAIL");

    // 清理
    free(h_input);
    cudaFree(d_input);
    cudaFree(d_partial_val);
    cudaFree(d_partial_idx);
    cudaFree(d_output);

    return 0;
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/12_argmax.cu -o /tmp/12_argmax.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“ArgMax 对归约与确定性”的工作机制。

**你的答案：**


### Q2

只用 `>` 时为什么不同 block 划分可能返回不同的相等最大值索引？

**你的答案：**


### Q3

若输入包含 NaN，NumPy、PyTorch 与你的 kernel 应采用哪种约定？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
%%writefile /tmp/12_argmax.cu
#include <cuda_runtime.h>
#include <float.h>
#include <stdio.h>
#include <stdlib.h>

namespace {

// ArgMax: 找到 input[0:n] 中最大值的索引。
//
// 面试重点：
// 1. reduce 时同时传递 (value, index) 对。
// 2. warp shuffle 传 value 比较，再传 index 选择。
// 3. 两轮 kernel：第一轮每个 block 出一个 (val, idx)，第二轮归并。

struct ValIdx {
    float val;
    int idx;
};

template<int BLOCK_SIZE>
__device__ __forceinline__ ValIdx warp_reduce_max(ValIdx vi) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset >>= 1) {
        float other_val = __shfl_down_sync(0xffffffff, vi.val, offset);
        int other_idx = __shfl_down_sync(0xffffffff, vi.idx, offset);
        if (other_val > vi.val || (other_val == vi.val && other_idx >= 0 && (vi.idx < 0 || other_idx < vi.idx))) {
            vi.val = other_val;
            vi.idx = other_idx;
        }
    }
    return vi;
}

template<int BLOCK_SIZE>
__device__ __forceinline__ ValIdx block_reduce_max(ValIdx vi) {
    constexpr int NUM_WARPS = (BLOCK_SIZE + 31) / 32;
    __shared__ float smem_val[NUM_WARPS];
    __shared__ int smem_idx[NUM_WARPS];

    int lane = threadIdx.x & 31;
    int warp = threadIdx.x >> 5;

    vi = warp_reduce_max<BLOCK_SIZE>(vi);

    if (lane == 0) {
        smem_val[warp] = vi.val;
        smem_idx[warp] = vi.idx;
    }
    __syncthreads();

    if (warp == 0) {
        vi.val = (lane < NUM_WARPS) ? smem_val[lane] : -FLT_MAX;
        vi.idx = (lane < NUM_WARPS) ? smem_idx[lane] : -1;
        vi = warp_reduce_max<BLOCK_SIZE>(vi);
    }

    return vi;
}

// 第一轮：每个 block 用 grid-stride loop 找到局部最大值及其索引
template<int BLOCK_SIZE>
__global__ void argmax_kernel(
    const float* __restrict__ input,
    float* __restrict__ partial_val,
    int* __restrict__ partial_idx,
    int n
) {
    int tid = threadIdx.x;
    int idx = blockIdx.x * blockDim.x + tid;
    int stride = blockDim.x * gridDim.x;

    // 每个线程先通过 grid-stride loop 找到自己负责的元素中的最大值
    ValIdx local = {-FLT_MAX, -1};
    for (int i = idx; i < n; i += stride) {
        if (input[i] > local.val) {
            local.val = input[i];
            local.idx = i;
        }
    }

    // block 级 reduce
    ValIdx result = block_reduce_max<BLOCK_SIZE>(local);

    if (tid == 0) {
        partial_val[blockIdx.x] = result.val;
        partial_idx[blockIdx.x] = result.idx;
    }
}

// 第二轮：把所有 block 的 partial 结果归并，只需一个 block
template<int BLOCK_SIZE>
__global__ void argmax_final_kernel(
    const float* __restrict__ partial_val,
    const int* __restrict__ partial_idx,
    int* __restrict__ output,
    int num_blocks
) {
    int tid = threadIdx.x;

    ValIdx local = {-FLT_MAX, -1};
    // 一个 block 处理所有 partial，stride = BLOCK_SIZE
    for (int i = tid; i < num_blocks; i += BLOCK_SIZE) {
        if (partial_val[i] > local.val) {
            local.val = partial_val[i];
            local.idx = partial_idx[i];
        }
    }

    ValIdx result = block_reduce_max<BLOCK_SIZE>(local);

    if (tid == 0) {
        *output = result.idx;
    }
}

} // namespace

// ===================== Host 调用接口 =====================

void launch_argmax(
    const float* d_input,
    int* d_output,
    float* d_partial_val,
    int* d_partial_idx,
    int n,
    int num_blocks,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    argmax_kernel<BLOCK_SIZE><<<num_blocks, BLOCK_SIZE, 0, stream>>>(
        d_input, d_partial_val, d_partial_idx, n
    );
    argmax_final_kernel<BLOCK_SIZE><<<1, BLOCK_SIZE, 0, stream>>>(
        d_partial_val, d_partial_idx, d_output, num_blocks
    );
}

// ===================== 测试 =====================

int main() {
    const int N = 1 << 20;  // 1M 元素
    const int NUM_BLOCKS = 128;

    // Host 数据
    float* h_input = (float*)malloc(N * sizeof(float));
    srand(42);
    int expected_idx = 0;
    float expected_val = -FLT_MAX;
    for (int i = 0; i < N; i++) {
        h_input[i] = (float)rand() / RAND_MAX;
        if (h_input[i] > expected_val) {
            expected_val = h_input[i];
            expected_idx = i;
        }
    }

    // Device 数据
    float *d_input, *d_partial_val;
    int *d_partial_idx, *d_output;
    cudaMalloc(&d_input, N * sizeof(float));
    cudaMalloc(&d_partial_val, NUM_BLOCKS * sizeof(float));
    cudaMalloc(&d_partial_idx, NUM_BLOCKS * sizeof(int));
    cudaMalloc(&d_output, sizeof(int));

    cudaMemcpy(d_input, h_input, N * sizeof(float), cudaMemcpyHostToDevice);

    // 执行
    launch_argmax(d_input, d_output, d_partial_val, d_partial_idx, N, NUM_BLOCKS, 0);

    // 取结果
    int h_output;
    cudaMemcpy(&h_output, d_output, sizeof(int), cudaMemcpyDeviceToHost);

    printf("ArgMax result: index = %d, value = %f\n", h_output, h_input[h_output]);
    printf("Expected:      index = %d, value = %f\n", expected_idx, expected_val);
    printf("%s\n", (h_output == expected_idx) ? "PASS" : "FAIL");

    // 清理
    free(h_input);
    cudaFree(d_input);
    cudaFree(d_partial_val);
    cudaFree(d_partial_idx);
    cudaFree(d_output);

    return 0;
}


### Q1 参考答案

每线程找局部 `(val,idx)`，warp/block combine 时按值选择，并用索引规则打破平局，最后二阶段合并。

### Q2 参考答案

判断时先检查本课不变量：必须定义 tie-break 和 NaN policy；否则并行归约顺序变化会令结果不确定。当前初版采用严格 `>`，相等时保留先到者。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：保证最小索引确定性会增加比较逻辑；若调用方只关心任一最大值可放宽。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。